# Classify ice with d-SEAMS in a notebook

Install the wheel, fetch the checksummed cubic mW trajectory, and read the seeded cage assignment, the CHILL+ labels and a topology fingerprint from its first frame. Runs on Colab with the wheel from PyPI.

In [ ]:
%pip install -q pydseamslib ase

In [ ]:
import pathlib
import subprocess
import sys
import tempfile
import urllib.request

url = "https://raw.githubusercontent.com/d-SEAMS/seams-core/5aecb1031b10d0978e24aea20bfe300a5fa83e46/input/fetch.py"
path = pathlib.Path("mW_cubic.lammpstrj")
with tempfile.TemporaryDirectory() as directory:
    fetch = pathlib.Path(directory) / "fetch.py"
    urllib.request.urlretrieve(url, fetch)
    subprocess.run([sys.executable, str(fetch), path.name, "--output-dir", "."], check=True)
print(path.stat().st_size, "bytes")

In [ ]:
from pydseams import Frame

frame = Frame.from_file(str(path), atom_type=1)
print(frame.cloud.nop, "molecules")
score = frame.seeded_affiliation(ring_adjacent=True)
print("hexagonal-cage molecules", sum(score.hc), "double-diamond", sum(score.ddc))

In [ ]:
counts = frame.chill_plus()
print(counts)

In [ ]:
fp = frame.fingerprint(hops=3)
print(fp.method, "classes:", len(fp.classes), "ring census:", list(fp.ringCensus))
lib = frame.topology_library("Ic", hops=3)
print(frame.classify_topology(lib, hops=3).counts)

Every molecule of the cubic lattice falls in one topology class and the library built from the frame names all of them. Point `Frame.from_file` at your own LAMMPS dump, or build a frame from ASE with `pydseams.from_ase(atoms, select=('O',))`, and the same calls apply. The [features how-to](https://python.dseams.info/howto/features.html) turns a trajectory into per-frame vectors for PyEMMA and deeptime, and `pydseams.adapters` carries an MDAnalysis analysis and an OVITO modifier.